In [8]:
import fiftyone as fo
import pandas as pd

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os
from fiftyone import ViewField as F
from torchvision.datasets import CocoDetection
from torch.utils.data import Dataset
from torch.utils.data import Dataset
from torch.utils.data import Dataset



In [ ]:
import fiftyone as fo

# First, check the name of the dataset
print(fo.list_datasets())

# Likely name is "open-images-v7-train-10", unless you specified another name
#fo.delete_dataset("open-images-dog-1box")
#fo.delete_dataset("open-images-v7-train-10")
#fo.delete_dataset("open-images-v7-train-1000")
#fo.delete_dataset("open-images-v7-validation-100")




In [ ]:
print(fo.list_datasets())

In [ ]:
# dataset
dataset = fo.zoo.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=["Dog","Cat"],
    only_matching=True,
    max_samples=1000,
    overwrite=True
)

In [ ]:
print(dataset)

In [ ]:
# save dataset
dataset.persistent = True
dataset.save()

In [ ]:
del dataset

In [ ]:
# load dataset
dataset = fo.load_dataset("open-images-v7-train-10")


In [ ]:
filtered = dataset.match(
    F("ground_truth.detections").length() == 1
)

In [ ]:
print(filtered)

In [ ]:
print("Filtered samples:", len(filtered))

# Save as a new dataset
filtered.clone(name="open-images-dog-cat-1box")

In [ ]:
filtered.export(
    export_dir="dogs_cats_1box",
    dataset_type=fo.types.COCODetectionDataset,
    label_field="ground_truth",
    overwrite=True
)

In [ ]:
import json

with open("dogs_cats_1box/labels.json") as f:
    coco_data = json.load(f)

# Build ID → label name mapping
categories = coco_data["categories"]
id_to_label = {cat["id"]: cat["name"] for cat in categories}

print(id_to_label)


In [ ]:
dataset = CocoDetection(
    root="dogs_cats_1box/data",
    annFile="dogs_cats_1box/labels.json"
)

# Print labels for each image
for idx in range(len(dataset)):
    _, targets = dataset[idx]
    labels = [id_to_label[obj['category_id']] for obj in targets]
    print(f"Image {idx} labels: {labels}")

In [ ]:
fo.delete_dataset("open-images-v7-train-1000")

In [5]:
dataset = fo.load_dataset("open-images-dog-cat-1box")

In [9]:
class CocoSingleLabelDataset(Dataset):
    def __init__(self, image_dir, annotation_file, transform=None):
        self.base = CocoDetection(image_dir, annotation_file)
        self.transform = transform

        with open(annotation_file) as f:
            coco_data = json.load(f)
        self.id_to_label = {cat['id']: cat['name'] for cat in coco_data['categories']}
        self.label_to_index = {name: i for i, name in enumerate(sorted(set(self.id_to_label.values())))}

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, annotations = self.base[idx]

        # Get the first object’s label (you filtered for 1-box images)
        category_id = annotations[0]['category_id']
        label_name = self.id_to_label[category_id]
        label = self.label_to_index[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label